# Executed run — dq_quarantine_sa1 / task `nemweb_ingest`

Databricks job run `645430909659288`, task run `683718195799403`, started 2026-11-09 01:40:04 UTC, result **SUCCESS**.

Outputs below are exactly what the run printed (exported from the Databricks Jobs API). Run page: https://fevm-serverless-stable-2b9lj3.cloud.databricks.com/?o=7474644340067319#job/741084016505266/run/683718195799403


# Pipeline 01 — NEMWEB Dispatch Price Ingest (live, every 5 minutes)

Polls AEMO NEMWEB `/REPORTS/CURRENT/DispatchIS_Reports/`, downloads any 5-minute
DispatchIS ZIPs not yet landed, parses the `DISPATCH,PRICE` and `DISPATCH,REGIONSUM`
records and appends them to `{catalog}.bronze.nemweb_dispatch_price`.

Job 34 (`13_nemweb_bronze_to_gold`) then applies data-quality rules and merges
valid rows into `{catalog}.gold.nem_prices_5min`.

The notebook exits with a JSON summary (files fetched, rows written, latest interval)
so every run leaves auditable evidence in the job run output.

In [1]:
import csv
import io
import json
import re
import zipfile
from dataclasses import dataclass
from datetime import datetime, timezone
from urllib.parse import urljoin

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from pyspark.sql.types import StringType, StructField, StructType, TimestampType

dbutils.widgets.text("catalog", "energy_copilot_catalog")
dbutils.widgets.text("max_files", "288")  # bootstrap window: 288 files = last 24h

CATALOG = dbutils.widgets.get("catalog")
if not re.fullmatch(r"[A-Za-z0-9_]+", CATALOG):
    raise ValueError(f"Invalid catalog name: {CATALOG!r}")
MAX_FILES = int(dbutils.widgets.get("max_files"))

BRONZE = f"{CATALOG}.bronze.nemweb_dispatch_price"
LISTING_URL = "https://www.nemweb.com.au/REPORTS/CURRENT/DispatchIS_Reports/"
FILE_PATTERN = re.compile(r'href="([^"]*?(PUBLIC_DISPATCHIS_\d{12}_\d+\.zip))"', re.IGNORECASE)
NEM_REGIONS = frozenset({"NSW1", "QLD1", "VIC1", "SA1", "TAS1"})
EXTRA_COLUMNS = {  # added to the original bronze schema; all raw strings
    "settlement_datetime": "Full SETTLEMENTDATE (market time, UTC+10) YYYY/MM/DD HH:MM:SS",
    "total_demand": "REGIONSUM.TOTALDEMAND (MW)",
    "available_generation": "REGIONSUM.AVAILABLEGENERATION (MW)",
    "net_interchange": "REGIONSUM.NETINTERCHANGE (MW)",
}

## Schema evolution — add the columns this ingest needs (idempotent)

In [2]:
existing_cols = {f.name for f in spark.table(BRONZE).schema.fields}
for col, comment in EXTRA_COLUMNS.items():
    if col not in existing_cols:
        spark.sql(f"ALTER TABLE {BRONZE} ADD COLUMNS ({col} STRING COMMENT '{comment}')")
        print(f"Added column {col}")

## Discover new files

In [3]:
def build_session() -> requests.Session:
    retry = Retry(total=4, backoff_factor=2, status_forcelist=(429, 500, 502, 503, 504))
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.headers.update({"User-Agent": "energy-copilot-nemweb-ingest/1.0"})
    return session


def list_remote_files(session: requests.Session) -> list[tuple[str, str]]:
    resp = session.get(LISTING_URL, timeout=30)
    resp.raise_for_status()
    found = {name: urljoin(LISTING_URL, href) for href, name in FILE_PATTERN.findall(resp.text)}
    return sorted(found.items())  # file names sort chronologically (timestamp in name)


session = build_session()
remote = list_remote_files(session)
landed = {r[0] for r in spark.sql(f"SELECT DISTINCT _source_file FROM {BRONZE}").collect()}
pending = [(name, url) for name, url in remote if name not in landed][-MAX_FILES:]
print(f"Remote files: {len(remote)} | already landed: {len(landed)} | to fetch: {len(pending)}")

Remote files: 575 | already landed: 287 | to fetch: 288


## Download and parse

In [4]:
@dataclass(frozen=True)
class PriceRecord:
    source_file: str
    settlement_datetime: str
    run_no: str
    region_id: str
    rrp: str
    intervention: str
    last_changed: str
    total_demand: str | None
    available_generation: str | None
    net_interchange: str | None


def parse_dispatchis(name: str, payload: bytes) -> list[PriceRecord]:
    """Parse DISPATCH,PRICE rows and enrich with DISPATCH,REGIONSUM using each record's I-header."""
    headers: dict[str, list[str]] = {}
    prices: list[dict[str, str]] = []
    regionsum: dict[tuple[str, str, str, str], dict[str, str]] = {}
    with zipfile.ZipFile(io.BytesIO(payload)) as zf:
        for member in zf.namelist():
            text = zf.read(member).decode("utf-8", errors="replace")
            for row in csv.reader(io.StringIO(text)):
                if len(row) < 4 or row[1] != "DISPATCH" or row[2] not in ("PRICE", "REGIONSUM"):
                    continue
                if row[0] == "I":
                    headers[row[2]] = row
                elif row[0] == "D" and row[2] in headers:
                    rec = dict(zip(headers[row[2]], row))
                    key = (rec.get("SETTLEMENTDATE"), rec.get("RUNNO"), rec.get("REGIONID"), rec.get("INTERVENTION"))
                    if row[2] == "PRICE":
                        prices.append(rec)
                    else:
                        regionsum[key] = rec
    out = []
    for p in prices:
        key = (p.get("SETTLEMENTDATE"), p.get("RUNNO"), p.get("REGIONID"), p.get("INTERVENTION"))
        rs = regionsum.get(key, {})
        out.append(PriceRecord(
            source_file=name,
            settlement_datetime=p.get("SETTLEMENTDATE", ""),
            run_no=p.get("RUNNO", ""),
            region_id=p.get("REGIONID", ""),
            rrp=p.get("RRP", ""),
            intervention=p.get("INTERVENTION", ""),
            last_changed=p.get("LASTCHANGED", ""),
            total_demand=rs.get("TOTALDEMAND"),
            available_generation=rs.get("AVAILABLEGENERATION"),
            net_interchange=rs.get("NETINTERCHANGE"),
        ))
    return out


records: list[PriceRecord] = []
failed_files: list[dict[str, str]] = []
for name, url in pending:
    try:
        resp = session.get(url, timeout=60)
        resp.raise_for_status()
        records.extend(parse_dispatchis(name, resp.content))
    except (requests.RequestException, zipfile.BadZipFile) as exc:
        failed_files.append({"file": name, "error": str(exc)[:200]})
        print(f"WARN failed {name}: {exc}")

print(f"Parsed {len(records)} price rows from {len(pending) - len(failed_files)} files")

Parsed 1440 price rows from 288 files


## Append to bronze

In [5]:
BRONZE_SCHEMA = StructType([
    StructField("_source_file", StringType()),
    StructField("_ingested_at", TimestampType()),
    StructField("i_type", StringType()),
    StructField("settlement_date", StringType()),
    StructField("run_no", StringType()),
    StructField("regionid", StringType()),
    StructField("rrp", StringType()),
    StructField("intervention", StringType()),
    StructField("lastchanged", StringType()),
    StructField("settlement_datetime", StringType()),
    StructField("total_demand", StringType()),
    StructField("available_generation", StringType()),
    StructField("net_interchange", StringType()),
])

ingested_at = datetime.now(timezone.utc).replace(tzinfo=None)
rows = [
    (r.source_file, ingested_at, "DISPATCH,PRICE", r.settlement_datetime[:10], r.run_no, r.region_id,
     r.rrp, r.intervention, r.last_changed, r.settlement_datetime, r.total_demand,
     r.available_generation, r.net_interchange)
    for r in records
]
if rows:
    spark.createDataFrame(rows, schema=BRONZE_SCHEMA).write.mode("append").saveAsTable(BRONZE)

latest = max((r.settlement_datetime for r in records), default=None)
summary = {
    "status": "success" if rows or not pending else "failed",
    "files_listed": len(remote),
    "files_fetched": len(pending) - len(failed_files),
    "files_failed": len(failed_files),
    "rows_written": len(rows),
    "regions": sorted({r.region_id for r in records} & NEM_REGIONS),
    "latest_settlement_datetime_aest": latest,
    "target": BRONZE,
}
print(json.dumps(summary, indent=2))
if pending and not rows:
    raise RuntimeError(f"Fetched {len(pending)} files but wrote no rows: {failed_files[:3]}")
dbutils.notebook.exit(json.dumps(summary))

{"status": "success", "files_listed": 575, "files_fetched": 288, "files_failed": 0, "rows_written": 1440, "regions": ["NSW1", "QLD1", "SA1", "TAS1", "VIC1"], "latest_settlement_datetime_aest": "2026/11/09 05:25:00", "target": "serverless_stable_2b9lj3_catalog.bronze.nemweb_dispatch_price"}